# GlucoRisk_AI — Batched Prediction

Verify batched inference preserves predictions and row order.

Synthetic educational data. The connected domain endpoint differs from the original Gaussian tabular simulator. Window/horizon unit: hours.

In [1]:
from pathlib import Path
import sys,json
import numpy as np
import pandas as pd
ROOT=Path.cwd()
if not (ROOT/'src').exists():ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
from healthcare_ai.domain_loader import load_recording
from healthcare_ai.domain_experiment import recording_paths,load_default_cohort,DEFAULT_WINDOW,DEFAULT_STRIDE,UNIT
from healthcare_ai.settings import FEATURES,TARGET
from healthcare_ai.splits import split
from healthcare_ai.pipeline import build
paths=recording_paths();record=load_recording(paths[0]);frame=load_default_cohort()
train,val,test=split(frame);model=build().fit(train[FEATURES],train[TARGET])


In [2]:
from healthcare_ai.batch_inference import predict_batches
from healthcare_ai.inference import predict
bundle={'model':model,'threshold':.5,'metadata':{}}
a=predict(bundle,val);b=predict_batches(bundle,val,3)
print('Equivalent:',np.allclose(a.probability,b.probability))
print(b.head().to_string(index=False))

Equivalent: True
 patient_id  probability  predicted_class
GLU-SIM-005     0.020935                0
GLU-SIM-005     0.978128                1
GLU-SIM-005     0.017181                0
GLU-SIM-005     0.014377                0
GLU-SIM-005     0.958847                1


## Interpretation

Simulator results do not establish clinical performance. Repeated rows from a patient remain in one partition. Use patient-cluster uncertainty and keep validation analyses separate from final holdout evaluation.